# Exploratory Data Analysis (EDA)

Purpose: Perform EDA on Rossmann store sales data.
- Find a reliable store and filter data
- Plot raw series
- Seasonal decomposition (trend/seasonal/residual)
- ADF stationarity test
- Analyze holiday/promo effects

In [6]:
%pip install -r ../requirements.txt

Note: you may need to restart the kernel to use updated packages.


c:\Users\mahin\OneDrive\Desktop\Projects\Ana_projects\Demand Forecasting + Deployed Dashboard\.venv\Scripts\python.exe: No module named pip


In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller

plt.style.use('seaborn-v0_8-whitegrid')
import warnings
warnings.filterwarnings('ignore')

ModuleNotFoundError: No module named 'pandas'

In [ ]:
# Load data
train = pd.read_csv('../data/train.csv', low_memory=False)
store = pd.read_csv('../data/store.csv', low_memory=False)

df = train.merge(store, on='Store', how='left')
df['Date'] = pd.to_datetime(df['Date'])

In [ ]:
# Find a store with minimal missing data and no long closure gaps
store_stats = df.groupby('Store').agg(
    total_days=('Date', 'count'),
    open_days=('Open', 'sum'),
    sales_nulls=('Sales', lambda x: x.isnull().sum())
)
store_stats['open_ratio'] = store_stats['open_days'] / store_stats['total_days']

# We pick a store that has 0 nulls, max total days, and highest open ratio (least closures)
best_stores = store_stats[(store_stats['sales_nulls'] == 0) & (store_stats['total_days'] >= 942)].sort_values(by='open_ratio', ascending=False)
best_store_id = best_stores.index[0]

print(f"Selected Store ID: {best_store_id}")
print("Stats for this store:")
print(best_stores.loc[best_store_id])

# Filter to this store
store_df = df[df['Store'] == best_store_id].copy()
store_df = store_df.sort_values('Date').reset_index(drop=True)
store_df.set_index('Date', inplace=True)

In [ ]:
# Plot raw daily sales
plt.figure(figsize=(15, 5))
plt.plot(store_df.index, store_df['Sales'])
plt.title(f'Raw Daily Sales for Store {best_store_id}')
plt.xlabel('Date')
plt.ylabel('Sales')
plt.show()

In [ ]:
# Seasonal decomposition (weekly seasonality, period=7)
decomposition = seasonal_decompose(store_df['Sales'], period=7, model='additive')
fig = decomposition.plot()
fig.set_size_inches(15, 10)
plt.show()

In [ ]:
# ADF Stationarity Test
print("Running ADF Test on Sales series...")
adf_result = adfuller(store_df['Sales'])
print(f"ADF Statistic: {adf_result[0]:.4f}")
print(f"p-value: {adf_result[1]:.4e}")

if adf_result[1] < 0.05:
    print("Conclusion: The series is stationary (reject null hypothesis).")
else:
    print("Conclusion: The series is not stationary (fail to reject null hypothesis).")

In [ ]:
# Check Promo and StateHoliday effects
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
sns.boxplot(data=store_df, x='Promo', y='Sales')
plt.title('Sales by Promo')

plt.subplot(1, 2, 2)
store_df['StateHoliday'] = store_df['StateHoliday'].astype(str)
sns.boxplot(data=store_df, x='StateHoliday', y='Sales')
plt.title('Sales by StateHoliday')

plt.tight_layout()
plt.show()

print("\nAverage Sales by Promo:")
print(store_df.groupby('Promo')['Sales'].mean())
print("\nAverage Sales by StateHoliday:")
print(store_df.groupby('StateHoliday')['Sales'].mean())

## Final Summary of EDA

*(Pending user execution...)*